# Donut selection 

Owner: **Chris  Suberlak** [@suberlak](https://github.com/lsst-ts/ts_aos_analysis/issues/new?body=@suberlak) <br>
Last Verified to Run: **2025-05-29** <br>
Software Versions:
  - `lsst_distrib`: **w_2025_16**

## Setup:

This notebook was run on https://summit-lsp.lsst.codes/ .


## Imports

In [ ]:
%matplotlib inline
import os 
from lsst.daf.butler import Butler
import matplotlib.pyplot as plt 
from astropy.visualization import ZScaleInterval
import numpy as np
from lsst.obs.lsst import LsstCam

os.environ["LSST_RESOURCES_NUM_WORKERS"] = "10"

## Show example stamps and an associated quality table

In [ ]:
dayObs = 20250529
seqNum = 180
butler = Butler('LSSTCam', collections=['LSSTCam/runs/quickLook'])
dataRefs = list(butler.registry.queryDatasets('donutStampsExtra',
                 where=f"instrument='LSSTCam' and visit.day_obs={dayObs} and visit.seq_num = {seqNum}",
                              collections=['LSSTCam/runs/quickLook']
                             ))

In [ ]:
# The donut quality table
# contains information about both intra and extra-focal donuts,
# but it is always saved under the extra-focal dataId
donutQualityTable = butler.get('donutQualityTable', dataId=dataRefs[0].dataId, 
                               collections=['LSSTCam/runs/quickLook']
                              )

donutStampsExtra = butler.get('donutStampsExtra', dataId=dataRefs[0].dataId, 
                              collections=['LSSTCam/runs/quickLook']
                              )
visit = dataRefs[0].dataId['visit']

In [ ]:
def plot_stamps(stamps, donutQualityTableIntraExtra, visit, ncols=5, nrows=5):
    """
    Plot a grid of donut image stamps with quality selection indicators.

    This function displays a grid of intra- or extra-focal image stamps.
    Each stamp is annotated with its index, and optionally outlined in red
    if it is not included in the `FINAL_SELECT` column of the corresponding
    quality table.

    Args:
        stamps (list): A list of `DonutStamp`-like objects. Each must have
            a `.stamp_im.image.array` attribute and a shared `metadata`
            dict with the 'DFC_TYPE' key indicating 'intra' or 'extra'.
        donutQualityTableIntraExtra (pandas.DataFrame): A combined table
            of donut quality results. Contains both intra- and extra-focal
            stamps. Must include 'DEFOCAL_TYPE' and 'FINAL_SELECT' columns.
        visit (int or str): Visit identifier used for plot title.
        ncols (int, optional): Number of columns in the grid. Defaults to 5.
        nrows (int, optional): Number of rows in the grid. Defaults to 5.

    Notes:
        - The function selects only those rows from the quality table
          matching the focal type of the input stamps.
        - If there are more stamps than available subplot cells,
          only the first `ncols * nrows` will be displayed.
        - Stamps not marked as `FINAL_SELECT` will be highlighted
          with a red rectangle.

    """
    # find out whether these are intra or extra-focal
    dfc_type = stamps.metadata["DFC_TYPE"]

    # select an appropriate subset of donutQualityTable,
    # since there's one table for intra / extra ...
    donutQualityTable = donutQualityTableIntraExtra[
        donutQualityTableIntraExtra["DEFOCAL_TYPE"] == dfc_type
    ]
    ncells = nrows * ncols
    if ncells < len(stamps):
        print("Warning: insufficient number of cells to plot all stamps")
        print(f"Using a subset of {ncells} / {len(stamps)}")

    fig, axs = plt.subplots(nrows, ncols, figsize=(2 * ncols, 2 * nrows))
    ax = np.ravel(axs)
    i = 0

    for stamp in stamps:
        image = stamp.stamp_im.image.array
        width = len(image)

        if i < len(ax):
            image = stamp.stamp_im.image.array
            ax[i].imshow(image, origin="lower")

            ax[i].text(100, 100, i, fontsize=14, color="white")

            # mark with red rectangle donuts if not in FINAL_SELECT list
            if not donutQualityTable["FINAL_SELECT"][i]:
                rect = plt.Rectangle(
                    (0, 0), width, width, linewidth=8, edgecolor="r", facecolor="none"
                )
                # Add the patch to the Axes
                ax[i].add_patch(rect)

            ax[i].set_xticks([])
            ax[i].set_yticks([])
            i += 1
    fig.subplots_adjust(hspace=0.05, wspace=0.05)

    if len(stamps) < len(ax):
        for i in range(len(stamps), len(ax)):
            ax[i].axis("off")

    fig.suptitle(f"{visit}")


In [ ]:
 plot_stamps(donutStampsExtra, donutQualityTable, visit)

## Plot stamps together with information whether they were selected in the `donutStampSelector` 

The information whether this was used for final selection is stored in `calcZernikesTask_config`:

In [ ]:
dataRefs = list(butler.registry.queryDatasets('calcZernikesTask_config', 
                 where=f"instrument='LSSTCam' and visit.day_obs={dayObs} and visit.seq_num = {seqNum}",
                              collections=['LSSTCam/runs/quickLook']
                             ))
len(dataRefs)

In [ ]:
calcZernikesConfig = butler.get('calcZernikesTask_config', dataId=dataRefs[1].dataId,
                                collections=['LSSTCam/runs/quickLook']) 

If `calcZernikesConfig` is not available, then plot all present columns:

In [ ]:
def plot_stamps_and_selection(stamps, donutQualityTableIntraExtra, visit,
                              calcZernikesConfig=None):
                            
    """
    Plot a vertical grid of donut image stamps with selection metric annotations.

    This function displays each donut stamp in its own subplot row, alongside
    selection-related metrics like signal-to-noise (SN), entropy, fraction of
    bad pixels, and maximum power gradient. Each metric is color-coded:
    green if the selection criterion is met, red otherwise. A red border is
    drawn around any stamp not marked as `FINAL_SELECT`.

    Args:
        stamps (list): A list of stamp objects, each with a 
            `.stamp_im.image.array` attribute and a shared `metadata` 
            dictionary with 'DFC_TYPE' key.
        donutQualityTableIntraExtra (pandas.DataFrame): Combined quality table 
            including both intra- and extra-focal donut metrics. Must include 
            'DEFOCAL_TYPE', 'SN', 'ENTROPY', 'FRAC_BAD_PIX',
            'MAX_POWER_GRAD', and corresponding *_SELECT columns, plus 'FINAL_SELECT'.
        visit (int or str): Visit identifier for the plot title.
        calcZernikesConfig (optional): Configuration object that indicates 
            which selection metrics to display. Each metric has a boolean 
            flag like `.selectWithSignalToNoise`.

    Notes:
        - Only stamps matching the focal type (`DFC_TYPE`) are displayed.
        - The number of rows is equal to the number of stamps, with 1 column.
        - Metric values are drawn next to each stamp image.
        - The top row includes column labels for the metrics.
        - Unselected stamps are visually marked for easy inspection.

    """
    dfc_type = stamps.metadata['DFC_TYPE']
    detectorKey = 'LSST BUTLER DATAID DETECTOR'
    detId = ''
    if detectorKey in list(stamps.metadata):
        detId = stamps.metadata[detectorKey]
    
    # select subset of donutQualityTable, since there's one table for intra / extra ... 
    donutQualityTable = donutQualityTableIntraExtra[donutQualityTableIntraExtra['DEFOCAL_TYPE'] == dfc_type]
    nrows=len(stamps)
    ncols=1
    ncells = nrows * ncols 
    if ncells < len(stamps):
        print('Warning: insufficient number of cells to plot all stamps')
        print(f'Using a subset of {ncells} / {len(stamps)}')
              
    fig,axs = plt.subplots(nrows, ncols, figsize=(2*ncols,2*nrows), dpi=150)
    ax = np.ravel(axs)
    i=0
    width = len(stamps[0].stamp_im.image.array)
    
    xpos_col0 = 1.2*width
    xpos_col1 = 2*width
    xpos_col2 = 2.75*width
    xpos_col3 = 3.5*width
    xpos_col4 = 4.5*width
    xpos = width/2
    ypos = width/2
    titlefont = 12
    for stamp in stamps:
        image = stamp.stamp_im.image.array
        if i < len(ax):
            ax[i].imshow(image, origin='lower')

            ax[i].text(xpos,ypos,i,fontsize=14, color='white')
            color = 'k'
            if calcZernikesConfig is not None:
                if calcZernikesConfig.donutStampSelector.value.selectWithSignalToNoise:
                    color='green' if   donutQualityTable['SN_SELECT'][i] else 'red'
           
            ax[0].text(1.15*xpos_col0, 2*ypos, 'SN', color=color, fontsize=titlefont)
            ax[i].text(xpos_col0,ypos,  f"{donutQualityTable['SN'][i]:.2f}", fontsize=14, color=color)
            if calcZernikesConfig is not None:
                if calcZernikesConfig.donutStampSelector.value.selectWithEntropy:
                    color='green' if   donutQualityTable['ENTROPY_SELECT'][i] else 'red'
            ax[0].text(0.95*xpos_col1, 2*ypos, 'ENTROPY', color=color, fontsize=titlefont)
            ax[i].text(xpos_col1,ypos,  f"{donutQualityTable['ENTROPY'][i]:.2f}", fontsize=14, color=color)

            if calcZernikesConfig is not None:
                if calcZernikesConfig.donutStampSelector.value.selectWithFracBadPixels:
                    color='green' if   donutQualityTable['FRAC_BAD_PIX_SELECT'][i] else 'red'
            ax[0].text(0.95*xpos_col2, 2*ypos, 'FRAC BAD \nPIX', color=color, fontsize=titlefont)
            fracBadPix = donutQualityTable['FRAC_BAD_PIX'][i]
            if fracBadPix == 0:
                ax[i].text(xpos_col2, ypos,  
                           f"{fracBadPix:.0f}", fontsize=14, color=color)
            else:
                ax[i].text(xpos_col2, ypos, 
                           f"{fracBadPix:.2f}", fontsize=14, color=color)

            if calcZernikesConfig is not None:
                if calcZernikesConfig.donutStampSelector.value.selectWithMaxPowerGrad:
                    color='green' if   donutQualityTable['MAX_POWER_GRAD_SELECT'][i] else 'red'

            maxPowerGrad = donutQualityTable['MAX_POWER_GRAD'][i]
            
            ax[0].text(0.95*xpos_col3, 2*ypos, 'MAX POWER \nGRAD', color=color, fontsize=titlefont)
            
            if abs(maxPowerGrad) > 1e-2:
                ax[i].text(xpos_col3, ypos,  f"{maxPowerGrad:.3f}", fontsize=14, color=color)
            elif abs(maxPowerGrad) > 1e-3:
                ax[i].text(xpos_col3, ypos,  f"{maxPowerGrad:.4f}", fontsize=14, color=color)
            elif abs(maxPowerGrad) > 1e-4:
                ax[i].text(xpos_col3, ypos,  f"{maxPowerGrad:.5f}", fontsize=14, color=color)
            elif abs(maxPowerGrad) > 1e-5:
                ax[i].text(xpos_col3, ypos,  f"{maxPowerGrad:.6f}", fontsize=14, color=color) 
                
            color='green' if   donutQualityTable['FINAL_SELECT'][i] else 'red'
            ax[0].text(1.0*xpos_col4, 2*ypos, 'FINAL \nSELECT', color=color, fontsize=titlefont)
            ax[i].text(xpos_col4, ypos,  f"{donutQualityTable['FINAL_SELECT'][i]:.2f}", fontsize=14, color=color)
            
            # mark with red rectangle donuts if not in FINAL_SELECT list 
            if not donutQualityTable['FINAL_SELECT'][i] : 
                rect = plt.Rectangle((0, 0), width, width, linewidth=8, edgecolor='r', facecolor='none')
                # Add the patch to the Axes
                ax[i].add_patch(rect)
                
    
            ax[i].set_xticks([])
            ax[i].set_yticks([])
            i += 1 
    fig.subplots_adjust(hspace=0.05, wspace=0.05)
    
    if len(stamps)<len(ax):
        for i in range(len(stamps), len(ax)):
            ax[i].axis('off')


    ax[0].set_title(f'{visit},{detId},\n {dfc_type}')




# Plot donut selection criteria

In [ ]:
dayObs = 20250529
seqNum = 180
butler = Butler('LSSTCam', collections=['LSSTCam/runs/quickLook'])
dataRefs = list(butler.registry.queryDatasets('donutStampsExtra',
                where=f"instrument='LSSTCam' and visit.day_obs={dayObs} and visit.seq_num = {seqNum}",
                             
                            ))

# The donut quality table
# contains information about both intra and extra-focal donuts,
# but it is always saved under the extra-focal dataId
donutQualityTable = butler.get('donutQualityTable', dataId=dataRefs[0].dataId, 
                              
                              )

donutStampsExtra = butler.get('donutStampsExtra', dataId=dataRefs[0].dataId, 
                 
                              )
donutStampsIntra = butler.get('donutStampsIntra', dataId=dataRefs[0].dataId, 
                    
                              )
visit = dataRefs[0].dataId['visit']


In [ ]:
len(donutQualityTable)

In [ ]:
donutQualityTable[:4]

In [ ]:
plot_stamps_and_selection(donutStampsIntra, donutQualityTable, visit,)

In [ ]:
plot_stamps_and_selection(donutStampsExtra, donutQualityTable, visit,)